In [ ]:
from google.colab import drive
drive.mount('/gdrive', force_remount=True)

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle

In [ ]:
!mv kaggle.json ~/.kaggle/

In [ ]:
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# === CICIDS2017 (MachineLearningCVE) — Kaggle + Merge + Limpieza (estilo UNSW) ===
# VERSIÓN CORREGIDA respecto a la original de TFM1:
#   1. Se CONSERVAN las columnas de IP/puerto/timestamp si el dataset
#      las trae (antes se eliminaban explícitamente si venían con
#      nombres estilo UNSW "srcip"/"dstip", y se detecta ahora también
#      el nombrado típico de CICFlowMeter "source_ip"/"destination_ip").
#      Necesarias para poder construir escenarios de agregación por
#      origen más adelante (ver backend/app/features/flow_aggregation.py
#      del proyecto MODEXRE).
#   2. Corregido un bug real en la celda de verificación final: se
#      llamaba a pd.read_csv(..., memory_low=False) -- parámetro que no
#      existe en pandas (el correcto es low_memory) y que provocaba un
#      TypeError al ejecutar esa celda.
# Crea: /content/MachineLearningCVE_full_clean_v2.csv con:
#   - attack_cat normalizado (ALLOWED)
#   - label binaria 0/1 en category (derivada SOLO de attack_cat)
#   - ORDEN FINAL: attack_cat, label, IP/puerto/timestamp (si existen), resto

!pip install -q kaggle pandas numpy tqdm

import os, glob, zipfile, gc
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# -------------------- CONFIG --------------------
KAGGLE_DATASET = "pshikk/cicids2017-untampered"
# ANTES: "shrutikeshri17/machinelearningcve" -- ese mirror es la
# versión "MachineLearningCSV" del CICIDS2017 oficial (78 features),
# que NO incluye Flow ID/Source IP/Destination IP/Timestamp (se
# confirmó en ejecución real: solo sobrevivía "destination_port").
# El CICIDS2017 oficial también se distribuye como
# "GeneratedLabelledFlows.zip" (85 features), que SÍ conserva esas
# columnas de identificación. "pshikk/cicids2017-untampered" se marca
# a sí mismo como "dataset original, sin modificar", por lo que es el
# candidato más probable a conservarlas -- pero no se ha podido
# verificar sin descargarlo. La celda de verificación de este mismo
# notebook ("Columnas de IP/puerto/timestamp conservadas: [...]")
# confirma el resultado real al ejecutarlo. Si sale vacía otra vez,
# buscar en kaggle.com "CICIDS2017 GeneratedLabelledFlows" y sustituir
# este valor por el slug del dataset que se encuentre.
RAW_DIR        = "/content/MachineLearningCVE_raw"
ZIP_DIR        = "/content"
OUT_CSV        = "/content/MachineLearningCVE_full_clean_v2.csv"

WINSOR_Q_LOW   = 0.001
WINSOR_Q_HIGH  = 0.999
# -----------------------------------------------

os.makedirs(RAW_DIR, exist_ok=True)

# ============================================================
# NORMALIZADOR attack_cat (MISMA TAXONOMÍA BASE + BruteForce)
# ============================================================
ALLOWED = {
    "Normal",
    "Fuzzers","Exploits","DoS","Reconnaissance","Generic","Analysis",
    "Shellcode","Backdoors","DDoS","PortScan","MitM",
    "BruteForce"
}

# Mapeo directo (casos exactos / variantes típicas)
MAP = {
    # Normal
    "Benign": "Normal",
    "BENIGN": "Normal",
    "normal": "Normal",
    "benign": "Normal",
    "0": "Normal",
    "False": "Normal",
    "false": "Normal",
    "None": "Normal",
    "nan": "Normal",
    "NaN": "Normal",
    "": "Normal",

    # PortScan
    "Port Scan": "PortScan",
    "Portscan": "PortScan",
    "portscan": "PortScan",
    "PortScan": "PortScan",

    # DDoS / DoS
    "Ddos": "DDoS",
    "ddos": "DDoS",
    "DDoS": "DDoS",
    "Dos": "DoS",
    "dos": "DoS",
    "DoS": "DoS",

    # MitM
    "MITM": "MitM",
    "mitm": "MitM",
    "MitM": "MitM",

    # Si viene genérico
    "Attack": "Generic",
    "attack": "Generic",
    "Unknown": "Generic",
}

def _map_by_patterns(x: str) -> str:
    """
    Normalización por patrones para CICIDS:
    - Mantiene binario, pero conserva BruteForce, PortScan, DoS, DDoS cuando aplica.
    - Si no encaja con ningún patrón conocido, se CONSERVA el nombre
      original tal cual (title-case), en vez de colapsarlo a "Generic".
      Así cualquier etiqueta nueva o inesperada del CSV mantiene su
      propia identidad como clase, en vez de perderse mezclada con
      otras categorías bajo "Generic".
    """
    if x is None:
        return "Normal"
    s = str(x).strip()
    if s == "":
        return "Normal"

    sl = s.lower()

    # Normal / benign
    if sl in {"benign", "normal", "0", "false", "no"}:
        return "Normal"
    if "benign" in sl or "normal" in sl:
        return "Normal"

    # Brute force (Patator / brute)
    # CICIDS suele tener: FTP-Patator, SSH-Patator, Web Attack – Brute Force
    if "patator" in sl or "brute force" in sl or "bruteforce" in sl or "brute" in sl:
        return "BruteForce"

    # PortScan
    if "portscan" in sl or "port scan" in sl:
        return "PortScan"

    # DDoS (primero ddos)
    if "ddos" in sl:
        return "DDoS"

    # DoS (incluye Hulk, GoldenEye, Slowloris, SlowHTTPTest…)
    # OJO: si ya era ddos, no entra aquí.
    if sl.startswith("dos ") or " dos " in f" {sl} " or "dos-" in sl or "dos_" in sl or sl.startswith("dos"):
        return "DoS"

    # Cualquier otra cosa (p.ej. "Web Attack – XSS", "Infiltration",
    # "Heartbleed", "Bot"...): se conserva tal cual, no se fuerza a
    # Generic. Se normaliza solo el formato (title case, sin dobles
    # espacios) por consistencia visual.
    return " ".join(s.split()).title()

def normalize_attack_cat_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()

    # 1) mapeo directo (rápido)
    s = s.replace(MAP)

    # 2) mapeo por patrones (CICIDS-friendly) -- ya conserva lo
    #    desconocido tal cual, no colapsa a Generic (ver _map_by_patterns)
    s = s.apply(_map_by_patterns)

    return s

def enforce_attackcat_label(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "attack_cat" not in df.columns:
        raise ValueError("Falta attack_cat.")
    df["attack_cat"] = normalize_attack_cat_series(df["attack_cat"])
    df["label"] = (df["attack_cat"] != "Normal").astype(int).astype("category")
    return df

# -------------------- 1) Descargar dataset Kaggle --------------------
print(f">> Descargando dataset: {KAGGLE_DATASET}")
!kaggle datasets download -d $KAGGLE_DATASET -p $ZIP_DIR --force

zip_candidates = glob.glob(os.path.join(ZIP_DIR, "*.zip"))
if not zip_candidates:
    raise FileNotFoundError("No se descargó ningún .zip.")

ZIP_PATH = next(
    (z for z in zip_candidates if "machinelearningcve" in z.lower()),
    max(zip_candidates, key=os.path.getmtime)
)
print("✓ ZIP:", ZIP_PATH)

# -------------------- 2) Descomprimir --------------------
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(RAW_DIR)

# -------------------- 3) Localizar CSV --------------------
base_path = os.path.join(RAW_DIR, "MachineLearningCVE")
if not os.path.exists(base_path):
    base_path = next(
        p for p in glob.glob(os.path.join(RAW_DIR, "**"), recursive=True)
        if os.path.isdir(p) and glob.glob(os.path.join(p, "*.csv"))
    )

csv_files = sorted(glob.glob(os.path.join(base_path, "*.csv")))
print(">> CSV encontrados:", len(csv_files))

# -------------------- 4) Cargar y unir --------------------
def robust_read_csv(path):
    # IMPORTANTE: se prueba "cp1252" ANTES que "utf-8". Este dataset
    # se generó originalmente en Windows con CICFlowMeter, y algunas
    # etiquetas (p.ej. "Web Attack – XSS") usan un guion largo (en
    # dash, U+2013) codificado en cp1252. Leerlo como "utf-8" primero
    # no siempre lanza una excepción -- el parser de pandas puede
    # "tener éxito" silenciosamente sustituyendo esos bytes por el
    # carácter de reemplazo "�" (U+FFFD), sin fallar nunca a la
    # siguiente codificación del try/except. Resultado observado en
    # ejecución real: 'Web Attack � Sql Injection', 'Web Attack � Xss'
    # en vez de los nombres correctos con "–".
    for enc in ("cp1252", "latin1", "utf-8"):
        try:
            df_try = pd.read_csv(path, low_memory=False, encoding=enc)
            # Verificación extra: si tras decodificar sigue habiendo
            # el carácter de reemplazo en alguna columna de texto,
            # esta codificación tampoco es la correcta -> probar la
            # siguiente en vez de darla por buena.
            obj_cols = df_try.select_dtypes(include="object").columns
            if any(df_try[c].astype(str).str.contains("\ufffd", na=False).any() for c in obj_cols):
                continue
            return df_try
        except Exception:
            continue
    return pd.read_csv(path, low_memory=False, engine="python")

dfs = []
for f in tqdm(csv_files, desc="Leyendo CSV MachineLearningCVE"):
    dfs.append(robust_read_csv(f))

df = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()

# -------------------- 5) Normalizar nombres de columnas --------------------
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.replace(" ", "_", regex=False)
    .str.replace("/", "_", regex=False)
    .str.replace("-", "_", regex=False)
    .str.lower()
)

# -------------------- 6) Detectar label textual --------------------
if "label" not in df.columns:
    alt = [c for c in df.columns if c in ("class", "attack", "target")]
    if not alt:
        raise ValueError("No se encontró columna label.")
    df = df.rename(columns={alt[0]: "label"})

df["label"] = df["label"].astype(str).str.strip()

# -------------------- 7) Crear attack_cat desde label textual --------------------
# Si el label es benign/normal -> Normal; si no, lo tratamos como nombre de ataque y lo normalizamos
lab_lower = df["label"].str.lower()
is_benign = (
    lab_lower.isin({"benign","normal","0","false","no"})
    | lab_lower.str.contains("benign", na=False)
    | lab_lower.str.contains("normal", na=False)
)

df["attack_cat"] = np.where(is_benign, "Normal", df["label"])
df = enforce_attackcat_label(df)

# Limpieza cosmética: algunas etiquetas de este mirror de Kaggle ya
# vienen con el carácter de reemplazo "�" incrustado en el propio CSV
# de origen (p.ej. "Web Attack � Xss" en vez de "Web Attack – Xss").
# No es recuperable probando otras codificaciones -- el byte original
# ya se perdió antes de que este CSV se subiera a Kaggle. Se sustituye
# por un guion simple para que el nombre de la clase sea legible, sin
# que afecte a su función como etiqueta (sigue siendo un valor
# consistente y distinto del resto).
df["attack_cat"] = df["attack_cat"].astype(str).str.replace("\ufffd", "-", regex=False)

# -------------------- 8) Limpieza numérica --------------------
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)

for c in tqdm(num_cols, desc="Imputando numéricas"):
    if df[c].isna().any():
        med = df[c].median()
        df[c] = df[c].fillna(0.0 if np.isnan(med) else med)

for c in tqdm(num_cols, desc="Winsorización"):
    try:
        lo, hi = df[c].quantile(WINSOR_Q_LOW), df[c].quantile(WINSOR_Q_HIGH)
        if lo < hi:
            df[c] = df[c].clip(lo, hi)
    except Exception:
        pass

for c in df.select_dtypes(include=["float64"]).columns:
    df[c] = pd.to_numeric(df[c], downcast="float")
for c in df.select_dtypes(include=["int64"]).columns:
    df[c] = pd.to_numeric(df[c], downcast="integer")

# -------------------- 9) Eliminar columnas con texto "Infinity" --------------------
cols_bad = [
    c for c in df.columns
    if df[c].dtype == object and df[c].astype(str).str.contains("Infinity", na=False).any()
]
df = df.drop(columns=cols_bad, errors="ignore")

# -------------------- 9) IP/puertos: SE CONSERVAN --------------------
# ANTES: se eliminaban aquí columnas srcip/dstip/sport/dsport "por
# seguridad". Se conservan ahora porque son necesarias para poder
# construir escenarios de agregación por origen (agg_distinct_dst_ports/
# agg_distinct_dst_hosts/agg_events_in_window del proyecto MODEXRE,
# ver backend/app/features/flow_aggregation.py). Se detectan de forma
# robusta por si el dataset trae los nombres típicos de CICFlowMeter
# ("Source IP" -> "source_ip" tras normalize_cols) en vez de los de
# UNSW ("srcip"), que es lo más probable en este dataset concreto.
_ip_port_candidates = [
    "srcip", "dstip", "sport", "dsport",
    "source_ip", "destination_ip", "source_port", "destination_port",
    "flow_id", "timestamp",
]
ip_port_cols_present = [c for c in _ip_port_candidates if c in df.columns]
print(f"[INFO] Columnas de IP/puerto/timestamp conservadas: {ip_port_cols_present}")
if not ip_port_cols_present:
    print("[AVISO] Este dataset no trae ninguna columna de IP/puerto/timestamp "
          "reconocible tras la normalización de nombres. Revisa manualmente "
          "los nombres de columna originales del CSV si esperabas tenerlas.")

# -------------------- 10) Reordenar columnas --------------------
other_cols = [c for c in df.columns if c not in ["attack_cat", "label"] + ip_port_cols_present]
df = df[["attack_cat", "label"] + ip_port_cols_present + other_cols]

# -------------------- 11) Guardar --------------------
df.to_csv(OUT_CSV, index=False, encoding="utf-8")
print("🎉 LISTO — Guardado →", OUT_CSV)
print("Shape final:", df.shape)

# -------------------- 12) Verificación --------------------
print("\n[VERIFICACIÓN FINAL]")
print("Normal con label=1  →", int(((df["attack_cat"]=="Normal")  & (df["label"].astype(int)==1)).sum()))
print("No-Normal con label=0 →", int(((df["attack_cat"]!="Normal") & (df["label"].astype(int)==0)).sum()))
print("\nattack_cat top20:")
print(df["attack_cat"].value_counts().head(20))
print("\nlabel:")
print(df["label"].value_counts())

In [ ]:
# === CICIDS2017 (MachineLearningCVE) — SDV (GaussianCopula) por cuotas + robusto en memoria === 04/01/2026
!pip install -q sdv packaging tqdm

import os, gc, time, math
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from collections import Counter
from sdv.metadata import Metadata
from sdv.single_table import GaussianCopulaSynthesizer

# ===================== CONFIG =====================
REAL_CSV  = "/content/MachineLearningCVE_full_clean_v2.csv"
SYN_CSV   = "/content/synthetic_MachineLearningCVE_ctgan_v2.csv"
META_JSON = "/content/MachineLearningCVE_metadata_gc_v2.json"

RANDOM_STATE = 42

# Cuotas finales sintéticas (ajústalas aquí)
NORMAL_N      = 200_000
ATTACK_N_EACH = 30_000   # por cada clase != Normal detectada (DoS, DDoS, PortScan, BruteForce, Generic...)

# Lectura por chunks (evita RAM llena)
CHUNK_SIZE = 250_000

# Muestra para entrenar SDV (forzando ataques)
TRAIN_TOTAL_N         = 250_000
TRAIN_NORMAL_MAX      = 140_000
TRAIN_ATTACK_MIN_EACH = 35_000

DEFAULT_DISTRIBUTION = "gamma"
ENFORCE_MINMAX = False
# ================================================

def normalize_cols(cols):
    return [str(c).strip().replace(" ", "_").replace("/", "_").replace("-", "_").lower() for c in cols]

def enforce_label_attackcat_inplace(df_any: pd.DataFrame) -> pd.DataFrame:
    if "attack_cat" not in df_any.columns:
        raise ValueError("Falta 'attack_cat' en el REAL. Revisa tu full_clean.")
    df_any["attack_cat"] = df_any["attack_cat"].astype(str).str.strip()
    df_any["label"] = (df_any["attack_cat"].str.lower() != "normal").astype(int)
    return df_any

def approx_line_count(path: str) -> int:
    with open(path, "rb") as f:
        return max(sum(1 for _ in f) - 1, 0)

def append_csv(df_part: pd.DataFrame, path: str):
    header = not os.path.exists(path)
    df_part.to_csv(path, index=False, mode="a", header=header)

# ===================== 1) PASADA 1: contar clases (sin cargar en RAM) =====================
n_lines = approx_line_count(REAL_CSV)
n_chunks = max(1, int(math.ceil(n_lines / CHUNK_SIZE)))
print(f"[INFO] REAL filas aprox: {n_lines:,} | chunks: {n_chunks} | chunksize: {CHUNK_SIZE:,}")

cnt_attack = Counter()
reader = pd.read_csv(REAL_CSV, low_memory=False, chunksize=CHUNK_SIZE)

for ch in tqdm(reader, total=n_chunks, desc="Scan clases (chunks)", unit="chunk"):
    ch.columns = normalize_cols(ch.columns.tolist())
    if "attack_cat" not in ch.columns:
        raise ValueError("Este CSV no tiene attack_cat.")
    cnt_attack.update(ch["attack_cat"].astype(str).str.strip().tolist())
    del ch
    gc.collect()

attack_cats = sorted([c for c in cnt_attack.keys() if str(c).strip().lower() != "normal"])
print("[INFO] attack_cat detectadas (sin Normal):", attack_cats)
print("[INFO] top10 real:", dict(Counter(cnt_attack).most_common(10)))

if not attack_cats:
    raise RuntimeError("No detecto clases de ataque. Revisa MachineLearningCVE_full_clean.csv.")

# ===================== 2) PASADA 2: construir df_train estratificado =====================
rng = np.random.RandomState(RANDOM_STATE)
train_parts = []
seen_per_cat = Counter()

reader = pd.read_csv(REAL_CSV, low_memory=False, chunksize=CHUNK_SIZE)

for ch in tqdm(reader, total=n_chunks, desc="Construyendo df_train (chunks)", unit="chunk"):
    ch.columns = normalize_cols(ch.columns.tolist())
    ch = enforce_label_attackcat_inplace(ch)

    # A) ataques: forzar mínimo por clase
    for cat in attack_cats:
        need = TRAIN_ATTACK_MIN_EACH - seen_per_cat[cat]
        if need <= 0:
            continue
        sub = ch[ch["attack_cat"] == cat]
        if len(sub) == 0:
            continue
        take = min(need, len(sub))
        samp = sub.sample(n=take, random_state=int(rng.randint(0, 1e9)))
        train_parts.append(samp)
        seen_per_cat[cat] += len(samp)

    # B) Normal: cap
    needN = TRAIN_NORMAL_MAX - seen_per_cat["Normal"]
    if needN > 0:
        subN = ch[ch["attack_cat"].str.lower() == "normal"]
        if len(subN) > 0:
            takeN = min(needN, len(subN))
            sampN = subN.sample(n=takeN, random_state=int(rng.randint(0, 1e9)))
            train_parts.append(sampN)
            seen_per_cat["Normal"] += len(sampN)

    del ch
    gc.collect()

    if sum(seen_per_cat.values()) >= TRAIN_TOTAL_N:
        break

df_train = pd.concat(train_parts, ignore_index=True)
del train_parts
gc.collect()

df_train = df_train.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
if len(df_train) > TRAIN_TOTAL_N:
    df_train = df_train.sample(n=TRAIN_TOTAL_N, random_state=RANDOM_STATE).reset_index(drop=True)

print("\n[INFO] df_train:", df_train.shape)
print("[INFO] df_train attack_cat:", df_train["attack_cat"].value_counts().to_dict())

# ===================== Exportar muestra manejable para MODEXRE =====================
# df_train es la muestra estratificada, no el CSV completo: es la que
# hay que subir a la pestaña Laboratorio de MODEXRE (ver misma nota en
# el notebook de UNSW-NB15 sobre el límite de subida de Streamlit).
TRAIN_SAMPLE_CSV = "/content/MachineLearningCVE_train_sample_v2.csv"
df_train.to_csv(TRAIN_SAMPLE_CSV, index=False, encoding="utf-8")
print(f"[OK] Muestra de entrenamiento exportada → {TRAIN_SAMPLE_CSV}")
print(f"     Peso aproximado: {os.path.getsize(TRAIN_SAMPLE_CSV) / (1024*1024):.1f} MB")
print("[INFO] df_train label:", df_train["label"].value_counts().to_dict())

# ===================== 3) SDV fit =====================
df_sdv = df_train.copy()
for c in df_sdv.columns:
    if str(df_sdv[c].dtype) == "category":
        df_sdv[c] = df_sdv[c].astype("object")

df_sdv["attack_cat"] = df_sdv["attack_cat"].astype(str)
df_sdv["label"] = df_sdv["label"].astype(str)

metadata = Metadata.detect_from_dataframe(df_sdv)
metadata.save_to_json(META_JSON)

synth = GaussianCopulaSynthesizer(
    metadata,
    default_distribution=DEFAULT_DISTRIBUTION,
    enforce_min_max_values=ENFORCE_MINMAX
)

print("\n[INFO] Entrenando sintetizador (fit) con muestra estratificada...")
t_fit = time.time()
synth.fit(df_sdv)
print(f"[OK] fit() terminado en {(time.time()-t_fit)/60:.2f} min")

# ===================== 4) Generar sintético por cuotas (incremental) =====================
if os.path.exists(SYN_CSV):
    os.remove(SYN_CSV)

# Normal
print("\n[INFO] Generando 'Normal' =", NORMAL_N)
t0 = time.time()
synN = synth.sample(num_rows=NORMAL_N)
synN.columns = normalize_cols(synN.columns.tolist())
synN = enforce_label_attackcat_inplace(synN)
synN["attack_cat"] = "Normal"
synN["label"] = 0
append_csv(synN, SYN_CSV)
del synN
gc.collect()
print(f"[OK] Normal guardado en {(time.time()-t0)/60:.2f} min")

# Ataques por clase
for cat in tqdm(attack_cats, desc="Generando ataques (por clase)", unit="clase"):
    synA = synth.sample(num_rows=ATTACK_N_EACH)
    synA.columns = normalize_cols(synA.columns.tolist())
    synA = enforce_label_attackcat_inplace(synA)
    synA["attack_cat"] = str(cat)
    synA["label"] = 1
    append_csv(synA, SYN_CSV)
    del synA
    gc.collect()

print("\n[OK] Sintético guardado en:", SYN_CSV)
print("[OK] Metadata guardada en:", META_JSON)

# ===================== 5) Validación rápida del SYN (sin cargar entero) =====================
cnt_syn_attack = Counter()
cnt_syn_label = Counter()
syn_reader = pd.read_csv(SYN_CSV, chunksize=200_000, low_memory=False)

for ch in tqdm(syn_reader, desc="Validando SYN (chunks)", unit="chunk"):
    cnt_syn_attack.update(ch["attack_cat"].astype(str).str.strip().tolist())
    cnt_syn_label.update(ch["label"].astype(str).str.strip().tolist())

print("\n[SYN] label:", dict(cnt_syn_label))
print("[SYN] attack_cat top15:", dict(Counter(cnt_syn_attack).most_common(15)))
print("\n[FIN] CICIDS_SYN listo (deberías ver BruteForce/PortScan/DoS/DDoS si existen en el REAL).")

In [ ]:
print("[REAL] attack_cat:", pd.read_csv("/content/MachineLearningCVE_full_clean_v2.csv", low_memory=False)["attack_cat"].value_counts().head(30))
print("[SYN]  attack_cat:", pd.read_csv("/content/synthetic_MachineLearningCVE_ctgan_v2.csv", low_memory=False)["attack_cat"].value_counts().head(30))

In [ ]:
from google.colab import files
files.download("/content/MachineLearningCVE_full_clean_v2.csv")
files.download("/content/MachineLearningCVE_train_sample_v2.csv")
files.download("/content/synthetic_MachineLearningCVE_ctgan_v2.csv")
files.download("/content/MachineLearningCVE_metadata_gc_v2.json")